# MTAM Reproduction — ZuCo Sentiment Analysis (sentence-level)

Runs the **forked** repo `parmisbathaeiyan/EEG_Language_Alignment`, branch `reproduction`,
instead of patching upstream code inside this notebook. Every change lives as a git
commit on the fork, so `git log upstream/main..reproduction` is the exact list of
deviations from the published code (printed in section 2).

Uses the **original ZuCo `.mat` data** with the upstream `prepare_sr_eeg_data` loader —
no custom dataloader. The sentiment-label CSV ships in the repo
(`preprocessed/ZuCo/sentiment_labels_clean.csv`), so only the `.mat` files come from Drive.

**Current experiment:** reproduce the paper's simplest EEG-only baseline, **MLP-EEG**,
with cross-entropy loss (Table 1 target: F1 0.480, accuracy 0.499). No text features,
Transformer encoder, or multimodal alignment loss are used.

**Branch state (commits over upstream):**
- compat (modern PyTorch/Colab) + reporting (labeled confusion matrix, macro P/R/F1, JSON/PNG)
- bug fixes: best-val checkpoint, complete test loader, seeded RNGs, configurable early stopping
- methodology fixes: 80/10/10 split, optional oversampling, standard scaled-dot-product attention
- EEG-only runs skip unrelated BERT construction and embedding work
- restored the released MLP class to the active training path

Data and results stay on Drive; `data/` is gitignored and never committed.

## 1. Mount Drive & configure paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verified against Drive on 2026-07-12. Folder contains results<SUBJ>_SR.mat files.
OG_ZUCO_SR_DIR = '/content/drive/MyDrive/Thesis/Data/zuco_og_raw'
RESULTS_ROOT   = '/content/drive/MyDrive/Thesis/Results/reproduce_EEG_Language_Alignment'
EEG_CACHE      = '/content/eeg_dict_cache.pkl'   # runtime cache; delete if source data changes

FORK_URL = 'https://github.com/parmisbathaeiyan/EEG_Language_Alignment.git'
BRANCH   = 'reproduction'

import os
assert os.path.isdir(OG_ZUCO_SR_DIR), f'SR .mat folder not found: {OG_ZUCO_SR_DIR}'
mats = sorted(f for f in os.listdir(OG_ZUCO_SR_DIR) if f.endswith('.mat'))
assert mats, f'No .mat files in {OG_ZUCO_SR_DIR}'
os.makedirs(RESULTS_ROOT, exist_ok=True)
print('Drive mounted. SR .mat files:', mats)

## 2. Clone the fork (reproduction branch)

In [ ]:
%cd /content
!rm -rf /content/EEG_Language_Alignment
!git clone --branch {BRANCH} {FORK_URL}
%cd /content/EEG_Language_Alignment
print('\nDeviations from upstream:')
!git remote add upstream https://github.com/Jason-Qiu/EEG_Language_Alignment.git 2>/dev/null; git fetch -q upstream
!git log --oneline upstream/main..{BRANCH}

## 3. Install dependencies
The repo's `requirements.txt` is frozen for Python 3.7 / CUDA 11.3. Install modern,
Colab-compatible versions instead.

In [ ]:
!pip install -q transformers==4.40.0
!pip install -q POT           # Python Optimal Transport (Wasserstein loss path)
!pip install -q scipy numpy pandas scikit-learn tqdm matplotlib seaborn
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 4. Prepare data
The upstream loader reads `data/SR/*.mat` and `data/sentiment_labels_clean.csv`.
Symlink the `.mat` files into `data/SR/`, copy the repo's label CSV, make output dirs.

In [ ]:
%cd /content/EEG_Language_Alignment
import os, shutil

os.makedirs('data/SR', exist_ok=True)
for d in ['lr_curves', 'pred_labels', 'baselines']:
    os.makedirs(d, exist_ok=True)

# Symlink the original .mat files into data/SR/ (no copy).
for f in os.listdir(OG_ZUCO_SR_DIR):
    if f.endswith('.mat'):
        dst = f'data/SR/{f}'
        if not os.path.exists(dst):
            os.symlink(os.path.join(OG_ZUCO_SR_DIR, f), dst)

# Label CSV ships in the repo.
shutil.copy('preprocessed/ZuCo/sentiment_labels_clean.csv', 'data/sentiment_labels_clean.csv')

print('data/SR:', sorted(os.listdir('data/SR')))
import pandas as pd
_df = pd.read_csv('data/sentiment_labels_clean.csv')
print('labels:', _df.shape, '| dist:', _df['sentiment_label'].value_counts().to_dict())

## 5. Run harness
Defines `run_experiment()` (run this cell once; it does not train anything by itself).
Logs stream to screen and to a `.txt`; a results `.json` and learning-curve `.png` are
written to Drive under `RESULTS_ROOT/<folder>/`.

In [ ]:
import subprocess, os, datetime

def run_experiment(modality, loss, folder_name, *,
                   model='transformer', level='sentence',
                   num_layers=1, num_heads=5, batch_size=64,
                   epochs=200, warm_steps=2000, dropout=0.3,
                   mlp_hidden_sizes=(256, 128, 64),
                   mlp_implementation='released', mlp_bias=0,
                   optimizer_type='scheduled_adam', lr=1e-5, weight_decay=1e-2,
                   eps=1e-4, adam_betas=(0.9, 0.98),
                   patience=20, oversample=1, suffix=''):
    ts = datetime.datetime.now().strftime('%Y%m%d%H%M%S')
    beta1, beta2 = adam_betas
    opt_tag = (f'{optimizer_type}_lr{lr:g}_wd{weight_decay:g}_eps{eps:g}'
               f'_betas{beta1:g}-{beta2:g}')
    if optimizer_type == 'scheduled_adam':
        opt_tag += f'_ws{warm_steps}'
    if model == 'MLP':
        hidden_tag = '-'.join(map(str, mlp_hidden_sizes))
        impl_tag = f'{mlp_implementation}_bias{mlp_bias}'
        run_name = f'{model}_{modality}_{level}_{loss}_h{hidden_tag}_{impl_tag}_b{batch_size}_{opt_tag}_p{patience}'
    else:
        run_name = f'{model}_{modality}_{level}_{loss}_L{num_layers}H{num_heads}b{batch_size}_{opt_tag}_p{patience}'
    run_name += f'_{suffix}' if suffix else ''
    run_dir  = os.path.join(RESULTS_ROOT, folder_name)
    os.makedirs(os.path.join(run_dir, 'logs'),  exist_ok=True)
    os.makedirs(os.path.join(run_dir, 'plots'), exist_ok=True)
    log_path  = os.path.join(run_dir, 'logs',  f'{run_name}_{ts}.txt')
    json_path = os.path.join(run_dir,          f'{run_name}_{ts}.json')
    plot_dst  = os.path.join(run_dir, 'plots', f'{run_name}_{ts}.png')

    cmd = ['python', '-u', 'main_new.py',
           '--dataset', 'ZuCo', '--task', 'SA', '--level', level,
           '--modality', modality, '--model', model, '--loss', loss,
           '--batch_size', str(batch_size), '--epochs', str(epochs),
           '--num_layers', str(num_layers), '--num_heads', str(num_heads),
           '--dropout', str(dropout), '--warm_steps', str(warm_steps),
           '--mlp_hidden_sizes', *map(str, mlp_hidden_sizes),
           '--mlp_implementation', mlp_implementation, '--mlp_bias', str(mlp_bias),
           '--optimizer_type', optimizer_type, '--lr', str(lr),
           '--weight_decay', str(weight_decay), '--eps', str(eps),
           '--adam_beta1', str(beta1), '--adam_beta2', str(beta2),
           '--patience', str(patience), '--oversample', str(oversample),
           '--eeg_cache', EEG_CACHE,
           '--inference', '0', '--dev', '0', '--device', 'cuda',
           '--timestamp', ts, '--json_path', json_path, '--plot_dst', plot_dst]

    print(f'Run: {run_name} | {ts}\nDir: {run_dir}\n' + '-'*60)
    with open(log_path, 'w') as lf:
        lf.write(f'Run: {run_name}\nTimestamp: {ts}\nArgs: {" ".join(cmd[1:])}\n' + '-'*60 + '\n')
        p = subprocess.Popen(cmd, cwd='/content/EEG_Language_Alignment',
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1,
                             env={**os.environ, 'TQDM_DISABLE': '1',
                                  'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'})
        for line in p.stdout:
            print(line, end=''); lf.write(line); lf.flush()
        return_code = p.wait()
    if return_code != 0:
        raise RuntimeError(f'Training failed with exit code {return_code}. See {log_path}')
    print('-'*60 + f'\nDone. Log -> {log_path}')

print('run_experiment() ready.')

## 6. Run the experiment — v11 MLP architecture sweep

Optimizer/regularization changes did not restore learning. This sweep returns to the best v9
training setup and tests the paper/code architecture discrepancy. It compares: (1) the historical
three-hidden shape 512/256/128 with bias enabled; (2) an inferred six-hidden interpretation
256/256/128/128/64/64 without bias; and (3) that inferred shape with bias enabled.

The released `MLP` class remains unchanged. The six-layer topology is explicitly our inference
from the paper's six-layer selection plus its listed widths 256/128/64; it is not claimed as
recovered author code. Paper target: macro F1 0.480, accuracy 0.499. Run the sweep once after
sections 1–5; all three results are saved separately.

In [ ]:
# Three controlled architecture diagnostics. Run this cell once.
common = dict(modality='eeg', model='MLP', loss='CE',
              folder_name='v11_MLP_EEG_architectureSweep', batch_size=32,
              mlp_implementation='configurable', optimizer_type='adam',
              lr=1e-3, weight_decay=1e-2, eps=1e-4,
              adam_betas=(0.9, 0.98), dropout=0.3,
              epochs=200, patience=20, oversample=1)

diagnostic_runs = [
    dict(mlp_hidden_sizes=(512, 256, 128), mlp_bias=1,
         suffix='historical3Hidden_withBias'),
    dict(mlp_hidden_sizes=(256, 256, 128, 128, 64, 64), mlp_bias=0,
         suffix='inferred6Hidden_noBias'),
    dict(mlp_hidden_sizes=(256, 256, 128, 128, 64, 64), mlp_bias=1,
         suffix='inferred6Hidden_withBias'),
]

for run in diagnostic_runs:
    run_experiment(**common, **run)

---
### After the sweep
Tell Codex all three runs have finished. We will compare their Drive JSONs against the existing
released three-hidden controls and paper target (F1 0.480 / accuracy 0.499), record the outcome
in `PROJECT_LOG.md`, and select the next factor. Do not rerun unless we explicitly decide to
replicate. To pick up later commits, re-run section 2.